# TAREA · Segmentación: lo que Orange no trae
Dos técnicas: **mezclas gaussianas (GMM)** y **k-prototypes** (datos mixtos). Ejecute las celdas en orden y responda las preguntas. Los archivos `clientes_retail.csv` y `formas_alargadas.csv` deben estar en la misma carpeta que el notebook.

Al final guardará sus resultados (`segmentos_notebook.csv` y los modelos en `.joblib`) para usarlos en su app de Streamlit.

In [1]:
!pip install kmodes -q
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
from kmodes.kprototypes import KPrototypes

## 1. Mezclas gaussianas: grupos alargados
k-means asume grupos redondos. GMM permite grupos con forma de elipse inclinada. Los datos son sintéticos y tenemos la respuesta real (`grupo_real`).

In [2]:
a = pd.read_csv('/content/formas_alargadas.csv')
a = a.rename(columns={'m#grupo_real':'grupo_real'})
Z = StandardScaler().fit_transform(a[['x','y']])
km = KMeans(3, n_init=10, random_state=0).fit_predict(Z)
gm = GaussianMixture(3, random_state=0, n_init=5).fit_predict(Z)
print('Acuerdo con la verdad (1 = perfecto)')
print('k-means:', round(adjusted_rand_score(a.grupo_real, km), 2))
print('GMM    :', round(adjusted_rand_score(a.grupo_real, gm), 2))

Acuerdo con la verdad (1 = perfecto)
k-means: 0.82
GMM    : 0.97


**Pregunta:** ¿por qué GMM acierta más? ¿Qué ventaja extra tiene (pista: da una probabilidad de pertenecer a cada grupo)?

In [3]:
probs = GaussianMixture(3, random_state=0, n_init=5).fit(Z).predict_proba(Z)
print('Clientes con duda (ninguna probabilidad > 0.9):', int((probs.max(axis=1) < 0.9).sum()), 'de', len(a))

Clientes con duda (ninguna probabilidad > 0.9): 5 de 450


## 2. K-prototypes: números y categorías juntos
k-means solo entiende números. K-prototypes combina distancia numérica con coincidencia de categorías (aquí `educacion` y `estado_civil`).

In [4]:
c = pd.read_csv('/content/clientes_retail.csv')
c = c.rename(columns={'m#id_cliente':'id_cliente','D#educacion':'educacion','D#estado_civil':'estado_civil','mD#respondio_ultima_campana':'respondio'})
num = ['ingreso_anual','gasto_total','compras_tienda','compras_web','compras_catalogo','visitas_web_mes','recencia_dias','hijos_pequenos','edad']
cat = ['educacion','estado_civil']
X = pd.concat([pd.DataFrame(StandardScaler().fit_transform(c[num]), columns=num), c[cat].reset_index(drop=True)], axis=1)
kp = KPrototypes(n_clusters=3, init='Cao', random_state=0, n_init=3)
c['seg_kproto'] = kp.fit_predict(X.values, categorical=[len(num), len(num)+1])
print(c.groupby('seg_kproto')[['ingreso_anual','gasto_total','hijos_pequenos','edad']].mean().round(1))
print(c.seg_kproto.value_counts())

            ingreso_anual  gasto_total  hijos_pequenos  edad
seg_kproto                                                  
0                 58018.6        669.5             0.2  55.7
1                 76080.7       1298.4             0.0  51.9
2                 33979.6         85.2             0.8  47.8
seg_kproto
2    1012
0     627
1     566
Name: count, dtype: int64


## 3. ¿Coinciden con k-means?

In [5]:
km3 = KMeans(3, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(c[num]))
print('Acuerdo k-means vs k-prototypes:', round(adjusted_rand_score(km3, c.seg_kproto), 2))
print(pd.crosstab(km3, c.seg_kproto, rownames=['k-means'], colnames=['k-prototypes']))

Acuerdo k-means vs k-prototypes: 0.99
k-prototypes    0    1     2
k-means                     
0               1  560     0
1               1    0  1012
2             625    6     0


**Preguntas finales:** ¿qué segmentación le daría a marketing y por qué? ¿Cambia el valor por segmento (gasto medio) según la técnica?

## 4. Guarde sus resultados
Se guarda la asignación de cada cliente (para el visor de la app) y los modelos ajustados con `joblib`.

In [6]:
import joblib
sc = StandardScaler().fit(c[num])
Zc = sc.transform(c[num])
gm_retail = GaussianMixture(3, random_state=0, n_init=5).fit(Zc)
c['seg_gmm'] = gm_retail.predict(Zc)
c['seg_kmeans'] = km3
# Cluster = la técnica que usted recomienda (cámbiela si prefiere otra)
c['Cluster'] = 'C' + (c['seg_kproto'] + 1).astype(str)
c.to_csv('segmentos_notebook.csv', index=False)
joblib.dump({'scaler': sc, 'modelo': gm_retail, 'variables': num}, 'modelo_gmm.joblib')
joblib.dump({'modelo': kp, 'variables_numericas': num, 'variables_categoricas': cat}, 'modelo_kproto.joblib')
print('Guardado: segmentos_notebook.csv, modelo_gmm.joblib, modelo_kproto.joblib')

Guardado: segmentos_notebook.csv, modelo_gmm.joblib, modelo_kproto.joblib
